# Optimising Adaptive Cancer Therapy under Stochastic Tumour Dynamics
**Stochastic Processes and Simulations in Natural Sciences — course project**  
**Authors:** *[Insert names]*  

**Research question:** Under what conditions can adaptive treatment delay tumour progression compared with continuous treatment in a stochastic model?

This notebook is a reproducible *in-silico* investigation, not a clinical recommendation. Its rates and time units are illustrative, not fitted to patient data. The course asks for extensively commented code, representative parameter regimes, systematic dependence on parameters, 5–10 figures, and conclusions. We provide **seven numbered figures** and a held-out evaluation of threshold optimisation.

## 1. Model and assumptions
We track sensitive cells $S$ and resistant cells $R$ in a well-mixed population, with total $N=S+R$. Cells divide at per-cell rates $b_S,b_R$. Density-dependent competition causes deaths at rates $c_S S N/K$ and $c_R R N/K$, and baseline deaths occur at rates $d_S S,d_R R$. Drug treatment $u\in\{0,1\}$ adds death rates $u k_S S$ and $u k_R R$. We assume $k_S>k_R$, so resistant cells are less affected. No mutations, spatial structure, pharmacokinetics, or treatment toxicity are modelled.

The six Gillespie reaction propensities are

| Event | State change | Propensity |
|---|---|---|
| Sensitive birth | $(S,R)\to(S+1,R)$ | $b_S S$ |
| Resistant birth | $(S,R)\to(S,R+1)$ | $b_R R$ |
| Sensitive death | $(S,R)\to(S-1,R)$ | $S(d_S+c_S N/K+u k_S)$ |
| Resistant death | $(S,R)\to(S,R-1)$ | $R(d_R+c_R N/K+u k_R)$ |

At total rate $a_0$, the waiting time is exponential with mean $1/a_0$; the reaction is selected in proportion to its propensity. A fixed policy switch can occur *before* the next random event: the simulation explicitly handles this to avoid incorrectly keeping the previous treatment state through a scheduled switch.

The corresponding mean-field ODE (not an exact equation for expected counts because of nonlinear competition) is
$$\dot S=S[b_S-d_S-c_S(S+R)/K-u k_S],\quad \dot R=R[b_R-d_R-c_R(S+R)/K-u k_R].$$

**Progression** is the *first time* $N\ge P$, not simply whether the tumour is above $P$ at the horizon. If both populations reach zero, they remain extinct. Drug exposure is $\int_0^T u(t)dt/T$ (fraction of time on drug).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, replace
from scipy.integrate import solve_ivp
from scipy.stats import beta
from IPython.display import display

SEED = 20261122
rng_master = np.random.default_rng(SEED)
plt.rcParams.update({'figure.figsize': (8, 4.6), 'axes.grid': True, 'grid.alpha': 0.25})

@dataclass(frozen=True)
class Params:
    bS: float = 0.85
    bR: float = 0.67     # cost of resistance in the absence of treatment
    dS: float = 0.15
    dR: float = 0.15
    cS: float = 0.70
    cR: float = 0.70
    K: float = 240.0
    kS: float = 0.95     # drug-sensitive kill rate
    kR: float = 0.07     # resistant cells are less affected
    S0: int = 145
    R0: int = 5
    P: int = 210         # progression threshold
    T: float = 70.0

par = Params()
print(par)

## 2. Treatment policies
- **Continuous:** drug always on.
- **Periodic:** on for the first 6 time units of every 12-unit cycle; otherwise off.
- **Adaptive:** drug turns on when total size reaches the upper threshold `hi` and turns off when it reaches the lower threshold `lo`. Between these thresholds the previous drug state persists (hysteresis). Drug begins **off** if $N_0<hi$ and **on** otherwise.

These choices are explicit modelling conventions. The adaptive policy uses exact integer population states; the ODE implementation uses continuous event detection.

In [ ]:
@dataclass(frozen=True)
class Policy:
    kind: str
    lo: int = 100
    hi: int = 160
    on_time: float = 6.0
    period: float = 12.0

def initial_drug(policy, total):
    if policy.kind == 'continuous': return 1
    if policy.kind == 'periodic': return 1
    if policy.kind == 'adaptive': return int(total >= policy.hi)
    raise ValueError(policy.kind)

def periodic_state(t, policy):
    # Right-continuous convention at switching times.
    return int((t % policy.period) < policy.on_time)

def next_periodic_switch(t, policy):
    # Calculate the next *strictly future* switching time.
    q = np.floor(t / policy.period)
    candidates = [q*policy.period + policy.on_time, (q+1)*policy.period,
                  (q+1)*policy.period + policy.on_time]
    return min(x for x in candidates if x > t + 1e-10)

def update_adaptive(u, N, policy):
    if N >= policy.hi: return 1
    if N <= policy.lo: return 0
    return u

def simulate(policy, p=par, seed=0, dt=0.5):
    # Exact event-driven CTMC with policy switching; sample on a regular grid.
    rng = np.random.default_rng(seed)
    grid = np.linspace(0, p.T, int(round(p.T/dt))+1)
    S, R = p.S0, p.R0
    t, exposure, progress_time = 0., 0., np.inf
    u = initial_drug(policy, S+R)
    if S+R >= p.P: progress_time = 0.
    outS = np.empty(len(grid)); outR = np.empty(len(grid)); outU = np.empty(len(grid))
    j = 0
    while t < p.T - 1e-12:
        # Record the state at every observation point before the next event.
        if policy.kind == 'periodic': u = periodic_state(t, policy)
        elif policy.kind == 'adaptive': u = update_adaptive(u, S+R, policy)
        N = S+R
        rates = np.array([p.bS*S, p.bR*R,
                          S*(p.dS+p.cS*N/p.K+u*p.kS),
                          R*(p.dR+p.cR*N/p.K+u*p.kR)], dtype=float)
        total = rates.sum()
        event_time = t + rng.exponential(1/total) if total > 0 else np.inf
        switch_time = next_periodic_switch(t, policy) if policy.kind == 'periodic' else np.inf
        tnext = min(event_time, switch_time, p.T)
        while j < len(grid) and grid[j] < tnext - 1e-10:
            outS[j], outR[j], outU[j] = S, R, u
            j += 1
        exposure += u*(tnext-t)
        t = tnext
        if t >= p.T - 1e-12: break
        if switch_time <= event_time:
            u = periodic_state(t + 1e-9, policy)
        else:
            event = rng.choice(4, p=rates/total)
            if event == 0: S += 1
            elif event == 1: R += 1
            elif event == 2: S -= 1
            else: R -= 1
            if S+R >= p.P and not np.isfinite(progress_time): progress_time = t
    while j < len(grid):
        outS[j], outR[j], outU[j] = S, R, u
        j += 1
    return dict(t=grid, S=outS, R=outR, N=outS+outR, u=outU,
                progress_time=progress_time, exposure=exposure/p.T)

policies = {'Continuous': Policy('continuous'),
            'Periodic': Policy('periodic'),
            'Adaptive (illustrative)': Policy('adaptive', lo=95, hi=160)}

# Sanity checks: no negative cells, bounded exposure, and reproducibility.
for name, pol in policies.items():
    a = simulate(pol, seed=123)
    assert np.all(a['S'] >= 0) and np.all(a['R'] >= 0)
    assert 0 <= a['exposure'] <= 1
    assert np.array_equal(a['N'], simulate(pol, seed=123)['N'])
print('Simulation checks passed.')

## Figure 1 — Representative trajectories
One realisation per policy illustrates the competition mechanism, but single runs are **not** sufficient to rank strategies.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.7), sharey=True)
for i, (name, pol) in enumerate(policies.items()):
    x = simulate(pol, seed=42)
    axs[i].plot(x['t'], x['S'], label='Sensitive')
    axs[i].plot(x['t'], x['R'], label='Resistant')
    axs[i].plot(x['t'], x['N'], color='black', label='Total', lw=1.7)
    axs[i].axhline(par.P, color='crimson', ls='--', label='Progression')
    axs[i].set_title(name); axs[i].set_xlabel('Time (model units)')
axs[0].set_ylabel('Cell count'); axs[0].legend(fontsize=8)
fig.suptitle('Figure 1. Example stochastic trajectories'); fig.tight_layout(); plt.show()

## Figure 2 — Variation between stochastic realisations
Use independent random seeds for each replicate. The grey lines show trajectories; a thick line shows the median. This reveals why a deterministic average alone is inadequate.

In [ ]:
n_examples = 24
fig, axs = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, (name, pol) in zip(axs, policies.items()):
    runs = [simulate(pol, seed=1000+i) for i in range(n_examples)]
    arr = np.array([z['N'] for z in runs]); tt = runs[0]['t']
    for y in arr: ax.plot(tt, y, color='grey', alpha=.16, lw=.8)
    ax.plot(tt, np.median(arr, axis=0), color='navy', lw=2, label='Median')
    ax.axhline(par.P, ls='--', color='crimson')
    ax.set_title(name); ax.set_xlabel('Time')
axs[0].set_ylabel('Total cells'); axs[0].legend()
fig.suptitle('Figure 2. Between-run variability'); fig.tight_layout(); plt.show()

## 3. Deterministic benchmark
We solve the mean-field equations with treatment-policy switching. For adaptive therapy, the ODE is integrated until a threshold event, then restarted with the opposite drug state. This avoids falsely switching at arbitrary plotting times.

In [ ]:
def ode_trajectory(policy, p=par, dt=.5):
    tgrid = np.linspace(0, p.T, int(round(p.T/dt))+1)
    values = np.empty((len(tgrid), 2)); values[0] = [p.S0, p.R0]
    u = initial_drug(policy, p.S0+p.R0)
    t = 0.; y = np.array([float(p.S0), float(p.R0)])
    segments = []
    for _ in range(5000):
        if t >= p.T - 1e-10: break
        if policy.kind == 'periodic':
            u = periodic_state(t+1e-9, policy)
            end = min(p.T, next_periodic_switch(t+1e-9, policy))
        else: end = p.T
        def f(_, z):
            s,r = z; N=s+r
            return [s*(p.bS-p.dS-p.cS*N/p.K-u*p.kS),
                    r*(p.bR-p.dR-p.cR*N/p.K-u*p.kR)]
        events = None
        if policy.kind == 'adaptive':
            threshold = policy.lo if u else policy.hi
            def crossing(_, z): return z[0]+z[1]-threshold
            crossing.terminal = True
            crossing.direction = -1 if u else 1
            events = crossing
        sol = solve_ivp(f, (t, end), y, events=events, dense_output=True,
                        rtol=1e-8, atol=1e-10, max_step=.25)
        stop = float(sol.t[-1])
        segments.append((t, stop, u, sol.sol))
        y = np.maximum(sol.y[:, -1], 0)
        t = stop
        if policy.kind == 'adaptive' and sol.t_events[0].size:
            u = 1-u
        elif policy.kind != 'periodic': break
    for j, tj in enumerate(tgrid):
        for start, end, state, dense in segments:
            if start-1e-9 <= tj <= end+1e-9:
                values[j] = dense(np.clip(tj, start, end)); break
    # Integrate drug exposure over exact segments.
    exposure = sum((end-start)*state for start,end,state,_ in segments)/p.T
    first = np.where(values.sum(axis=1) >= p.P)[0]
    return dict(t=tgrid, S=values[:,0], R=values[:,1], N=values.sum(axis=1),
                exposure=exposure, progress_time=(tgrid[first[0]] if len(first) else np.inf))

# Test continuous-policy ODE against direct solve_ivp.
test = ode_trajectory(Policy('continuous'))
assert np.isfinite(test['N']).all()
print('ODE checks passed.')

## Figure 3 — Mean-field trajectories against stochastic ensembles
The ODE is a large-population approximation, not the mean of the exact nonlinear stochastic model. We compare it with stochastic median and central 80% range.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.7), sharey=True)
for ax, (name, pol) in zip(axs, policies.items()):
    arr = np.array([simulate(pol, seed=3000+i)['N'] for i in range(80)])
    x = ode_trajectory(pol)
    ax.fill_between(x['t'], *np.quantile(arr, [.1, .9], axis=0), alpha=.25, label='10–90% stochastic')
    ax.plot(x['t'], np.median(arr, axis=0), label='Stochastic median')
    ax.plot(x['t'], x['N'], '--', color='black', lw=2, label='ODE')
    ax.axhline(par.P, color='crimson', ls=':')
    ax.set_title(name); ax.set_xlabel('Time')
axs[0].set_ylabel('Total cells'); axs[0].legend(fontsize=8)
fig.suptitle('Figure 3. Deterministic vs stochastic dynamics'); fig.tight_layout(); plt.show()

## 4. Evaluation metrics and uncertainty
For each replicate, progression-free survival at time $t$ is $\mathbf{1}(	au_P>t)$, where $	au_P$ is the first threshold crossing. We estimate the survival curve by averaging indicators. Confidence intervals for the horizon probability use the exact binomial (Clopper–Pearson) method; intervals quantify Monte Carlo uncertainty **conditional on the chosen model parameters**, not biological uncertainty.

In [ ]:
def batch(policy, p, seeds):
    return [simulate(policy, p, int(s)) for s in seeds]

def summary(runs, horizon):
    successes = sum(x['progress_time'] > horizon for x in runs)
    n = len(runs)
    low = 0. if successes == 0 else beta.ppf(.025, successes, n-successes+1)
    high = 1. if successes == n else beta.ppf(.975, successes+1, n-successes)
    return dict(p_free=successes/n, ci=(low, high), exposure=np.mean([x['exposure'] for x in runs]))

# A distinct seed stream for every independent analysis.
seedgen = np.random.default_rng(SEED)
def new_seeds(n): return seedgen.integers(0, 2**32-1, size=n)

base_seeds = new_seeds(160)
base_runs = {name: batch(pol, par, base_seeds) for name, pol in policies.items()}
for name, runs in base_runs.items():
    print(name, summary(runs, par.T))

## Figure 4 — Probability of remaining progression-free over time
Once a run crosses the progression threshold it is counted as progressed forever, even if treatment later reduces its size.

In [ ]:
fig, ax = plt.subplots()
times = next(iter(base_runs.values()))[0]['t']
for name, runs in base_runs.items():
    firsts = np.array([z['progress_time'] for z in runs])
    surv = np.mean(firsts[:, None] > times[None, :], axis=0)
    ax.step(times, surv, where='post', label=name)
ax.set(xlabel='Time', ylabel='Estimated probability progression-free', ylim=(0, 1.02),
       title='Figure 4. Progression-free probability')
ax.legend(); plt.show()

## 5. Optimising adaptive thresholds — training set only
We search threshold pairs on a fixed grid, with $lo<hi<P$. The objective is estimated probability of remaining progression-free at $T$. To avoid selecting an unnecessarily drug-intensive strategy when estimates tie, we use **lower mean exposure as a tie-breaker**. All pairs are evaluated using the same training seeds (common random numbers); this reduces noise in comparisons, but selected performance is still optimistically biased. We therefore evaluate the winner on a *fresh*, untouched seed set.

In [ ]:
lower_grid = np.array([60, 85, 110, 135])
upper_grid = np.array([145, 160, 175, 190])
train_seeds = new_seeds(75)
score = np.full((len(lower_grid), len(upper_grid)), np.nan)
exposure_grid = np.full_like(score, np.nan)
for i, lo in enumerate(lower_grid):
    for j, hi in enumerate(upper_grid):
        if lo >= hi: continue
        z = summary(batch(Policy('adaptive', lo=int(lo), hi=int(hi)), par, train_seeds), par.T)
        score[i,j], exposure_grid[i,j] = z['p_free'], z['exposure']
valid = [(score[i,j], -exposure_grid[i,j], int(lower_grid[i]), int(upper_grid[j]))
         for i in range(len(lower_grid)) for j in range(len(upper_grid)) if np.isfinite(score[i,j])]
best = max(valid)
chosen = Policy('adaptive', lo=best[2], hi=best[3])
print('Selected adaptive thresholds:', chosen.lo, chosen.hi)
print('Training progression-free estimate:', best[0], 'exposure:', -best[1])

## Figure 5 — Threshold grid-search heatmap
A threshold combination that looks best on training simulations is not necessarily best in the true model distribution. The next section performs held-out testing.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
im = ax.imshow(np.ma.masked_invalid(score), origin='lower', aspect='auto', vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(range(len(upper_grid)), upper_grid)
ax.set_yticks(range(len(lower_grid)), lower_grid)
ax.set(xlabel='Upper switch-on threshold', ylabel='Lower switch-off threshold',
       title='Figure 5. Training progression-free probability')
for i in range(len(lower_grid)):
    for j in range(len(upper_grid)):
        if np.isfinite(score[i,j]): ax.text(j, i, f'{score[i,j]:.2f}', ha='center', va='center', color='white', fontsize=9)
ax.plot(np.where(upper_grid==chosen.hi)[0][0], np.where(lower_grid==chosen.lo)[0][0],
        's', mfc='none', mec='red', mew=2, ms=27)
fig.colorbar(im, ax=ax, label='Training probability'); fig.tight_layout(); plt.show()

## 6. Held-out test and initial resistance sensitivity
The selected thresholds are **frozen**. We compare continuous, periodic and adaptive policies on independent evaluation seeds. We then vary initial resistant fraction while keeping the initial total count fixed, to identify regimes where the relative advantage changes. The selected thresholds are *not retuned* across resistance fractions.

In [ ]:
test_seeds = new_seeds(350)
final_policies = {'Continuous': Policy('continuous'), 'Periodic': Policy('periodic'),
                  'Adaptive (selected)': chosen}
heldout = {name: summary(batch(pol, par, test_seeds), par.T)
           for name, pol in final_policies.items()}
print('HELD-OUT evaluation:')
for name, result in heldout.items():
    print(f"{name:20s}  P(no progression)={result['p_free']:.3f} "
          f"95% CI=({result['ci'][0]:.3f}, {result['ci'][1]:.3f}) "
          f"mean drug exposure={result['exposure']:.3f}")

fractions = np.array([0.0, .02, .05, .10, .20, .35])
sensitivity = {name: [] for name in final_policies}
sens_seeds = new_seeds(130)
for frac in fractions:
    r0 = int(round((par.S0+par.R0)*frac))
    modified = replace(par, R0=r0, S0=par.S0+par.R0-r0)
    for name, pol in final_policies.items():
        sensitivity[name].append(summary(batch(pol, modified, sens_seeds), modified.T)['p_free'])

## Figure 6 — Dependence on initial resistant fraction
The same model parameters and policies are used, except for the initial split between sensitive and resistant cells. This is a sensitivity analysis, not a new training set.

In [ ]:
fig, ax = plt.subplots()
for name, values in sensitivity.items(): ax.plot(fractions, values, 'o-', label=name)
ax.set(xlabel='Initial resistant fraction', ylabel='Probability progression-free at T',
       ylim=(-.03,1.03), title='Figure 6. Sensitivity to initial resistance')
ax.legend(); plt.show()

## Figure 7 — Tumour control versus treatment exposure
Each point represents a held-out policy. Larger progression-free probability is better; smaller drug exposure is better. Exposure here measures treatment *duration*, not cumulative administered dose, toxicity, or pharmacodynamic effect.

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for name, z in heldout.items():
    ax.errorbar(z['exposure'], z['p_free'],
                yerr=[[z['p_free']-z['ci'][0]], [z['ci'][1]-z['p_free']]],
                fmt='o', capsize=4, markersize=8, label=name)
    ax.annotate(name, (z['exposure'], z['p_free']), xytext=(6,5), textcoords='offset points')
ax.set(xlabel='Mean fraction of time on treatment', ylabel='Progression-free probability at T',
       xlim=(-.04,1.07), ylim=(-.04,1.07),
       title='Figure 7. Held-out tumour control vs drug exposure')
plt.show()

## 7. Does deterministic optimisation select the same thresholds?
The following optional calculation compares the **deterministic** grid optimum (progression-free at the horizon, with lower exposure as tie-breaker) with the **stochastic** training optimum. Because the deterministic outcome is binary at a fixed horizon, this ranking can have many ties. This is a meaningful limitation: deterministic predictions do not directly yield progression probabilities. A more refined extension would optimise deterministic *time to progression* or minimum distance to the threshold, then compare held-out stochastic results.

In [ ]:
ode_candidates = []
for lo in lower_grid:
    for hi in upper_grid:
        if lo >= hi: continue
        pol = Policy('adaptive', lo=int(lo), hi=int(hi))
        x = ode_trajectory(pol)
        ode_candidates.append((int(x['progress_time'] > par.T), -x['exposure'], int(lo), int(hi)))
ode_best = max(ode_candidates)
print('ODE grid selection (binary survival, exposure tie-break):', (ode_best[2], ode_best[3]))
print('Stochastic grid selection:', (chosen.lo, chosen.hi))
print('Selections coincide:', (ode_best[2], ode_best[3]) == (chosen.lo, chosen.hi))

## 8. Interpretation and conclusions
The following cell generates a **data-dependent summary** rather than assuming adaptive treatment must win. Interpret differences in light of the binomial confidence intervals and the fact that all parameters are illustrative.

**Mechanism:** treatment disproportionately removes sensitive cells. When sensitive cells are depleted, resistant cells may face less competition, allowing resistant regrowth. Adaptive treatment preserves sensitive competitors during off-drug periods, but may also allow total tumour growth; its success depends on growth rates, drug selectivity, switching thresholds, initial resistance and stochastic fluctuations.

**Limitations:** no mutations or switching phenotypes, no spatial niches, no drug toxicity, no individual patient calibration, no imperfect measurement or delays, and a relatively small threshold grid. The Gillespie model captures demographic randomness only. In particular, a stochastic strategy's apparent advantage can reverse with parameter changes; these findings are not medical advice.

**Possible follow-ups:** (i) explore carrying capacity and drug selectivity; (ii) increase Monte Carlo repetitions and use bootstrap confidence intervals for differences between strategies; (iii) incorporate mutation or a measurement delay; (iv) optimise a continuous objective such as expected restricted progression-free time rather than a single-horizon binary objective.

In [ ]:
ranked = sorted(heldout.items(), key=lambda x: x[1]['p_free'], reverse=True)
winner, metrics = ranked[0]
print('MODEL-SPECIFIC CONCLUSION')
print(f'At the selected baseline parameters and T={par.T:g}, the largest held-out '
      f'progression-free estimate belongs to {winner}: {metrics["p_free"]:.3f} '
      f'(95% binomial CI {metrics["ci"][0]:.3f}–{metrics["ci"][1]:.3f}).')
for name, z in heldout.items():
    print(f'  {name}: progression-free={z["p_free"]:.3f}; exposure={z["exposure"]:.3f}')
print('Compare confidence intervals and examine Figure 6 before generalising.')
print('Adaptive thresholds were chosen on training seeds and assessed on fresh test seeds.')

## References and reproducibility
1. Course project instructions, *Stochastic Processes and Simulations in Natural Sciences — Projects* (provided PDF). The brief requests a well-commented notebook, 5–10 figures, systematic parameter study, and conclusions.
2. Gillespie, D. T. (1977). “Exact stochastic simulation of coupled chemical reactions.” *The Journal of Physical Chemistry* **81**, 2340–2361. DOI: [10.1021/j100540a008](https://doi.org/10.1021/j100540a008).
3. Gatenby, R. A., Silva, A. S., Gillies, R. J., & Frieden, B. R. (2009). “Adaptive Therapy.” *Cancer Research* **69**, 4894–4903. DOI: [10.1158/0008-5472.CAN-08-3658](https://doi.org/10.1158/0008-5472.CAN-08-3658).

The model in this notebook is a simplified teaching construction inspired by the competition rationale of adaptive therapy, **not** a reproduction or validation of any cited paper's specific model. All random number generator seeds are fixed. Required packages: `numpy`, `scipy`, `matplotlib`, `ipython`, `nbformat` (for notebook creation only). Use **Kernel → Restart & Run All** to reproduce the figures and tables. Increase the simulation counts for final submission if runtime permits.